In [ ]:
!pip install -U transformers datasets peft bitsandbytes accelerate
!pip install pandas openpyxl bert-score sentencepiece

In [2]:
import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model

In [21]:
def load_excel_dataset(path):
    df = pd.read_excel(path)
    df = df.head(20000)

    df["severity_probs"] = df[["غير حرج", "متوسط", "حرج"]].values.tolist()

    return Dataset.from_pandas(df)

In [4]:
def build_prompt(row):
    return f"""
{row['Question']}
"""


In [5]:
def tokenize(example):
    prompt = build_prompt(example)
    full_text = prompt + "\nالاجابة:\n" + example["Answer"]

    tokens = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()
    tokens["severity_probs"] = example["severity_probs"]

    return tokens

In [6]:
class SoftSeverityTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        severity_probs = inputs.pop("severity_probs").to(
        device=model.device,
        dtype=torch.float
    )


        outputs = model(**inputs)
        logits = outputs.logits

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = inputs["labels"][:, 1:].contiguous()

        log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)

        # Map severity → token weights (simple scaling)
        severity_weight = (
            severity_probs[:, 0] * 0.5 +
            severity_probs[:, 1] * 1.0 +
            severity_probs[:, 2] * 1.5
        ).unsqueeze(1).unsqueeze(2)

        loss_fct = torch.nn.NLLLoss(reduction="none")
        loss = loss_fct(
            log_probs.view(-1, log_probs.size(-1)),
            shift_labels.view(-1)
        )

        loss = loss.view(shift_labels.size())
        loss = loss * severity_weight
        loss = loss.mean()

        return (loss, outputs) if return_outputs else loss

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = ""

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

In [ ]:
from huggingface_hub import login
login('')

In [ ]:
model_name = "atlasia/Al-Atlas-0.5B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train_dataset = load_excel_dataset("")
train_dataset = train_dataset.map(tokenize, remove_columns=train_dataset.column_names)

In [19]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
training_args = TrainingArguments(
    output_dir="",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False
)


from transformers import default_data_collator

trainer = SoftSeverityTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=default_data_collator
)


# trainer.train()
trainer.train(
    resume_from_checkpoint=""
)
trainer.save_model("")
tokenizer.save_pretrained("")

In [10]:
def generate_answer(row):
    prompt = build_prompt(row)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.7
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
import pandas as pd
from bert_score import score
from tqdm.auto import tqdm # Import tqdm

# Load evaluation set
eval_df = pd.read_excel("")
output_path = ""


# Optional: limit rows for testing
eval_df = eval_df.head(2000)

preds = []
refs = []
questions = []

# Generate responses
for i, (_, row) in enumerate(tqdm(eval_df.iterrows()), start=1):
    pred = generate_answer(row)
    pred = pred.split(row["Question"], 1)[-1].strip()
    pred = pred.split("الاجابة:", 1)[-1].strip()

    preds.append(pred)
    refs.append(row["Answer"])
    questions.append(row["Question"])

    # 🔹 Save every 100 rows
    if i % 100 == 0:
        output_df = pd.DataFrame({
            "Question": questions,
            "Answer": refs,
            "Model_Response": preds
        })
        output_df.to_excel(output_path, index=False)


# Save Question, Answer, and Model Response
output_df = pd.DataFrame({
    "Question": questions,
    "Answer": refs,
    "Model_Response": preds
})
output_df.to_excel(output_path, index=False)

# Compute overall BERTScore (dataset-level)
P, R, F1 = score(
    preds,
    refs,
    lang="ar",
    verbose=True
)

overall_f1 = F1.mean().item()
print("Overall BERTScore F1:", overall_f1)